### Problem:
Running `batch_labeling.py` multiple times on Philadelphia kept resulting in:

📊 Label Distribution:
**oracle_label**
- Ambiguous        931
- Contradictory    316
- Answerable        31

### Problem 1.5:
**Improvement:** Changed Solve logic, because previously, for a landmark to be "found," it must satisfy a strict intersection:$$\text{Candidate} = (\text{Distance} \leq 1500m) \cap (\text{OSM\_Tags} \in \text{LANDMARK\_GROUPS})$$
So if a landmark in Philadelphia is 200m away (well within your 1500m horizon) but its OpenStreetMap data is missing the specific amenity or shop tags we defined in config.py, the result of that intersection is Zero.

**Why it's still a problem:** Even after the change, running `batch_labeling.py` on Philadelphia resulted in:

📊 Label Distribution:
**oracle_label**

- Ambiguous        931
- Contradictory    21
- Answerable       135

which is **impossible**.

In [ ]:
import pandas as pd

# 1. Load the results from the batch run
results_path = "../data/philadelphia/philadelphia_silver_standard.parquet"
df_results = pd.read_parquet(results_path)

# 2. Filter for only the Ambiguous rows
ambiguous_df = df_results[df_results['oracle_label'] == 'Ambiguous']

print(f"Total Ambiguous rows found: {len(ambiguous_df)}")

# 3. Display the first 10 instructions and their extracted nouns
# This will show us exactly what the NLP model "saw"
print(ambiguous_df[['instruction', 'extracted_noun', 'candidate_count']].head(10))

# 4. Grab a specific one to test in our diagnostic tool
sample_row = ambiguous_df.iloc[0]
test_instruction = sample_row['instruction']
start_node = sample_row['start_node']

print(f"\n🚀 Ready to test Sample ID: {sample_row['sample_id']}")

Total Ambiguous rows found: 931
                                          instruction extracted_noun  \
0   Meet to the west of you, at Ben & Jerry's ice ...           None   
1   Meet me at the cafe north of you on the north ...           None   
2   Meet me at the historic memorial on the south ...           None   
3   Go south and a bit east. You'll find me at the...           None   
4   I am at the American Eagle Outfitters which is...           None   
7   Move near the river to see me at the bench on ...           None   
9   Meet me at a post box east of you on the south...           None   
11  Meet me at the playground by the southeast cor...           None   
12  I'm at the fast food restaurant on the west si...           None   
13  Meet me at the bicycle parking on the south si...           None   

    candidate_count  
0                69  
1                47  
2                77  
3                25  
4                45  
7                25  
9                 2  

There it is! We just found the Root Cause of the 931 Ambiguous rows.

## 🕵️ The "None" Noun Diagnosis
Look at extracted_noun column: It is **100% None.**

Because the noun extraction is failing to identify the landmark name (e.g., it missed "Ben & Jerry's" or "American Eagle Outfitters"), your SymbolicSolver is falling back to a Categorical-only search.

### Example of a Possible Chain Reaction for Instruction 0:

- Instruction: "Meet to the west of you, at Ben & Jerry's..."

- Extraction: Returns category: FOOD, noun: None.

- Solver: Says, "I don't have a specific name, so find me every POI that matches the category FOOD within 1500m."

- The Result: It finds 69 different food places nearby.

- The Final Label: Since 69 > 1, the solver marks it as **Ambiguous.**

In [ ]:
import os
import sys

# Move up one level from the 'notebooks' folder to the project root
project_root = os.path.dirname(os.path.abspath("")) 
if project_root not in sys.path:
    sys.path.append(project_root)

# Now we can import our project modules
import config
from src.extraction_utils import CategoricalMatcher
import re

print("✅ Project modules linked successfully!")

✅ Project modules linked successfully!


In [5]:
from src.extraction_utils import CategoricalMatcher
matcher = CategoricalMatcher()
import re

def extract_rvs_target_fixed(text: str) -> tuple:
    text_clean = text.replace("’", "'").replace(" ,", ",")

    # 1. Anchor Search
    anchor_pattern = r"\b(at|me at|is at|to)\b\s+(.*)"
    match = re.search(anchor_pattern, text_clean, re.IGNORECASE)
    if not match: return "UNKNOWN", "UNKNOWN"
    
    span = match.group(2)

    # 2. Clipping (REMOVED 'at' and 'the' from here)
    stops = [
        r"\b(?:on|near|across|which|is|south|north|west|east|corner|end|middle)\b",
        r",", r"\."
    ]
    
    earliest_stop = len(span)
    for stop_pattern in stops:
        s_match = re.search(stop_pattern, span, re.IGNORECASE)
        if s_match and s_match.start() < earliest_stop:
            earliest_stop = s_match.start()
    
    noun = span[:earliest_stop].strip()

    # 3. Cleanup (Handle the leading/trailing noise here instead)
    noun = re.sub(r"^(the|a|an)\s+", "", noun, flags=re.IGNORECASE)
    
    category = matcher.get_category(noun)
    return category, noun.strip()

In [6]:
test_cases = [
    "Meet to the west of you, at Ben & Jerry's ice cream.",
    "I am at the American Eagle Outfitters which is south.",
    "Meet me at the cafe north of you"
]

for t in test_cases:
    cat, noun = extract_rvs_target_fixed(t)
    print(f"Input: {t}")
    print(f"Output -> Category: {cat} | Noun: {noun}\n")

Input: Meet to the west of you, at Ben & Jerry's ice cream.
Output -> Category: UNKNOWN | Noun: the

Input: I am at the American Eagle Outfitters which is south.
Output -> Category: CLOTHES | Noun: American Eagle Outfitters

Input: Meet me at the cafe north of you
Output -> Category: CAFE | Noun: cafe



In [7]:
def extract_rvs_target_v3(text: str) -> tuple:
    # 1. Clean
    text_clean = text.replace("’", "'").replace(" ,", ",")

    # 2. THE FIX: Find the LAST 'at' or 'to' before the landmark
    # This ignores "at the west" and finds "at Ben & Jerry's"
    potential_anchors = [m.start() for m in re.finditer(r"\b(at|to|me at|is at)\b", text_clean, re.IGNORECASE)]
    
    if not potential_anchors:
        return "UNKNOWN", "UNKNOWN"
    
    # We take the last anchor found in the sentence
    start_idx = potential_anchors[-1]
    # Move the pointer past the anchor word itself (e.g., skip 'at ')
    span = re.sub(r"^(at|to|me at|is at)\s+", "", text_clean[start_idx:], flags=re.IGNORECASE)

    # 3. Clipping (No 'at' or 'the' here!)
    stops = [
        r"\b(?:on|near|across|which|is|south|north|west|east|corner|end|middle)\b",
        r",", r"\."
    ]
    
    earliest_stop = len(span)
    for stop_pattern in stops:
        s_match = re.search(stop_pattern, span, re.IGNORECASE)
        if s_match and s_match.start() < earliest_stop:
            earliest_stop = s_match.start()
    
    noun = span[:earliest_stop].strip()

    # 4. Cleanup
    noun = re.sub(r"^(the|a|an)\s+", "", noun, flags=re.IGNORECASE)
    
    # 5. Resolve Category
    category = matcher.get_category(noun)
    return category, noun.strip()

In [8]:
test_cases = [
    "Meet to the west of you, at Ben & Jerry's ice cream.",
    "I am at the American Eagle Outfitters which is south.",
    "Meet me at the cafe north of you"
]

for t in test_cases:
    cat, noun = extract_rvs_target_v3(t)
    print(f"Input: {t}")
    print(f"Result -> Category: {cat} | Noun: {noun}\n")

Input: Meet to the west of you, at Ben & Jerry's ice cream.
Result -> Category: SHOP | Noun: Ben & Jerry's ice cream

Input: I am at the American Eagle Outfitters which is south.
Result -> Category: CLOTHES | Noun: American Eagle Outfitters

Input: Meet me at the cafe north of you
Result -> Category: CAFE | Noun: cafe



Found desired output using `extract_rvs_target_v3`. Updating it in `extraction_utils.py`.
But tested it and again got an **impossible** result:
📊 Label Distribution:
**oracle_label**
- Ambiguous        666
- Contradictory    455
- Answerable       157

In [18]:
import pandas as pd
import os
import numpy as np
import pickle
import config
import re

# 1. Ensure project root and imports are accessible
# (Assumes you are in project_root/notebooks/)
import sys
project_root = os.path.dirname(os.path.abspath(""))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.oracle_engine import OracleEngine
from src.extraction_utils import extract_rvs_target # This will use the file version (v3)

# 2. Setup Distance Helper
def notebook_haversine(lat1, lon1, lat2, lon2):
    R = 6371000 
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1 
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    return R * (2 * np.arcsin(np.sqrt(a)))

# --- START THE DIAGNOSTIC ---
try:
    # 3. Load Graph and Oracle (The Missing Pieces)
    config.CURRENT_CITY = 'philadelphia'
    graph_path = os.path.join(project_root, config.get_graph_path())
    poi_path = os.path.join(project_root, config.get_poi_path())
    
    print(f"🔄 Loading Graph from {graph_path}...")
    with open(graph_path, 'rb') as f:
        G = pickle.load(f)
        
    print(f"🔄 Initializing Oracle from {poi_path}...")
    oracle = OracleEngine(G, poi_path)

    # 4. Load the Ambiguous results
    results_path = os.path.join(project_root, "data", "philadelphia", "philadelphia_silver_standard.parquet")
    df_results = pd.read_parquet(results_path)
    ambiguous_samples = df_results[df_results['oracle_label'] == 'Ambiguous'].head(15)
    
    print(f"\n🔬 Analyzing {len(ambiguous_samples)} Ambiguous samples...\n" + "="*60)

    for idx, row in ambiguous_samples.iterrows():
        instr = row['instruction']
        s_node = row['start_node']
        
        # A. Extraction
        cat, noun = extract_rvs_target(instr) 
        
        # B. Solver Simulation
        start_data = G.nodes[s_node]
        start_coords = (start_data['y'], start_data['x'])
        tags = config.LANDMARK_GROUPS.get(cat, {})
        
        # Use the INSTANCE 'oracle', not the CLASS 'OracleEngine'
        candidates = oracle.resolve_nearby_candidates(
            tags, start_coords[0], start_coords[1], 
            radius_m=1500,
            landmark_name=noun
        )
        
        print(f"ID {idx} | Instruction: {instr[:50]}...")
        print(f"  ↳ Extracted: ({cat}, '{noun}')")
        
        if len(candidates) > 1:
            dists = sorted([notebook_haversine(start_coords[0], start_coords[1], c['coords'][0], c['coords'][1]) for c in candidates])
            d1, d2 = dists[0], dists[1]
            
            print(f"  ↳ Found {len(candidates)} candidates.")
            print(f"  ↳ Distance Gap: Closest={d1:.1f}m | Second={d2:.1f}m")
            
            if d1 < 200 and d2 > 500:
                print("  ↳ 💡 INSIGHT: Nearest-Neighbor would resolve this to Answerable.")
        elif len(candidates) == 0:
            print("  ↳ ❌ FAILED: Zero candidates found.")
        else:
            print("  ↳ ✅ SUCCESS: Correctly resolved to 1 node.")
        print("-" * 60)

except Exception as e:
    print(f"❌ Execution Error: {e}")
    import traceback
    traceback.print_exc()

🔄 Loading Graph from c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\philadelphia\philadelphia_graph.gpickle...


C:\Users\adan\AppData\Local\Temp\ipykernel_18424\158848522.py:35: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


🔄 Initializing Oracle from c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\philadelphia\philadelphia_poi.pkl...
📍 Extracting coordinates from philadelphia 'centroid' column...

🔬 Analyzing 15 Ambiguous samples...
ID 0 | Instruction: Meet to the west of you, at Ben & Jerry's ice crea...
  ↳ Extracted: (SHOP, 'Ben & Jerry's ice cream')
  ↳ Found 69 candidates.
  ↳ Distance Gap: Closest=171.5m | Second=416.7m
------------------------------------------------------------
ID 1 | Instruction: Meet me at the cafe north of you on the north side...
  ↳ Extracted: (CAFE, 'cafe')
  ↳ Found 47 candidates.
  ↳ Distance Gap: Closest=333.0m | Second=359.2m
------------------------------------------------------------
ID 2 | Instruction: Meet me at the historic memorial on the south side...
  ↳ Extracted: (MONUMENT, 'historic memorial')
  ↳ Found 77 candidates.
  ↳ Distance Gap: Closest=341.6m | Second=412.5m
------------------------------------------------------------
ID 3 | Instr

Conclusion:
## 🧩 The "Philadelphia Problem": Why 666 Rows Were Ambiguous

The latest test in Philadelphia resulted in **666 Ambiguous** labels and **455 Contradictions**. By analyzing the data, we discovered that our AI was "too smart" for its own good; it was looking at every single landmark in the city, while the human who wrote the instructions only cared about the ones right in front of them, as stated in the RVS paper (RVS being the dataset we're using)

### The Proposed Fix: The "Salience Filter"
Implementing a **Salience Filter** to mimic human focus:

* **The "Cafe" Explosion:** If an instruction says "Meet at the cafe," and there are 47 cafes within 1.5km, the AI used to give up (Ambiguous). Now, it assumes the **nearest** cafe is the target, especially if it's within 200m.

* **The "North-ish" Problem:** Humans are bad at angles. If a user says "Go North" but the building is actually "North-West," our old code called it a "Contradiction." We now use a **45° Directional Wedge**, allowing for human error.

* **The "Chatty" Instructions:** When a user says "the recycling place and let's save the planet," we now use **Hard Boundary Clipping** to ignore the "fluff" and focus only on the word "recycling."

In [24]:
# 1. Force reload of all modules
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import pickle
import config
import src.utils as utils

# Ensure project root is in path
project_root = os.path.dirname(os.path.abspath(""))
if project_root not in sys.path:
    sys.path.append(project_root)

# Import the updated classes/functions
from src.oracle_engine import OracleEngine
from src.symbolic_solver import SymbolicSolver
from src.extraction_utils import extract_rvs_target

# 2. Re-initialize with updated logic
config.CURRENT_CITY = 'philadelphia'
with open(os.path.join(project_root, config.get_graph_path()), 'rb') as f:
    G = pickle.load(f)

oracle = OracleEngine(G, os.path.join(project_root, config.get_poi_path()))
solver = SymbolicSolver(oracle)

# 3. Test the "Salience Filter" on known problematic IDs
# We will use ID 0 (Ben & Jerry's) and ID 11 (Playground)
test_ids = [0, 11]
df_phil = pd.read_parquet(os.path.join(project_root, "data/philadelphia/philadelphia_silver_standard.parquet"))

print(f"🚀 Testing Salience Filter & 45° Wedge Logic...\n" + "="*60)

for tid in test_ids:
    row = df_phil.iloc[tid]
    instr = row['instruction']
    start_node = row['start_node']
    
    # This now calls your updated solver.solve() which includes the Distance Ratio Test
    result = solver.solve(instr, start_node)
    
    print(f"ID {tid} | Instruction: {instr[:60]}...")
    print(f"  ↳ New State: {result['state']}")
    
    if result['state'] == 'Answerable':
        print(f"  ↳ ✅ SUCCESS: The Salience Filter resolved the ambiguity!")
    elif result['state'] == 'Ambiguous':
        print(f"  ↳ 🚩 STILL AMBIGUOUS: The landmarks were likely too close together.")
    else:
        print(f"  ↳ ℹ️ Result: {result['state']}")
    print("-" * 60)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


C:\Users\adan\AppData\Local\Temp\ipykernel_18424\2765541898.py:25: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


📍 Extracting coordinates from philadelphia 'centroid' column...
✅ Solver Initialized: Found 1 isolated graph components.
🚀 Testing Salience Filter & 45° Wedge Logic...
DEBUG: Solver extracted Noun: 'Ben & Jerry's ice cream'
ID 0 | Instruction: Meet to the west of you, at Ben & Jerry's ice cream on South...
  ↳ New State: Answerable
  ↳ ✅ SUCCESS: The Salience Filter resolved the ambiguity!
------------------------------------------------------------
DEBUG: Solver extracted Noun: 'playground by the southeast'
ID 11 | Instruction: Meet me at the playground by the southeast corner on Lanier ...
  ↳ New State: Answerable
  ↳ ✅ SUCCESS: The Salience Filter resolved the ambiguity!
------------------------------------------------------------


Success!
📊 Label Distribution:
**oracle_label**
- Answerable       1035
- Contradictory     161
- Ambiguous          82

Now optimizing (if needed):

In [25]:
# --- THE "TRUE ERROR" INVESTIGATOR ---
# Filter for what's left in the Ambiguous and Contradictory buckets
remaining_ambiguous = df_results[df_results['oracle_label'] == 'Ambiguous'].head(5)
remaining_contradictory = df_results[df_results['oracle_label'] == 'Contradictory'].head(5)

print(f"🕵️ Investigating 'Hard Failures' (True Errors)...\n")

print("--- 🚩 REMAINING AMBIGUOUS (The 'Twin' Problem) ---")
for idx, row in remaining_ambiguous.iterrows():
    res = solver.solve(row['instruction'], row['start_node'])
    print(f"ID {idx} | Noun: '{res['noun']}' | Candidates: {res['candidate_count']}")
    print(f"  ↳ Instruction: {row['instruction'][:80]}...")
    # These usually fail because d1 and d2 are too close (e.g., two Starbucks 50m apart)

print("\n--- ❌ REMAINING CONTRADICTORY (The 'Phantom' Problem) ---")
for idx, row in remaining_contradictory.iterrows():
    res = solver.solve(row['instruction'], row['start_node'])
    print(f"ID {idx} | Noun: '{res['noun']}' | Candidates: {res['candidate_count']}")
    if res['candidate_count'] == 0:
        print(f"  ↳ Reason: Landmark not found in OSM (Extraction or Data Gap).")
    else:
        print(f"  ↳ Reason: Directional mismatch or Reachability (Dead end).")
    print(f"  ↳ Instruction: {row['instruction'][:80]}...")

🕵️ Investigating 'Hard Failures' (True Errors)...

--- 🚩 REMAINING AMBIGUOUS (The 'Twin' Problem) ---
ID 0 | Noun: 'Ben & Jerry's ice cream' | Candidates: 69
  ↳ Instruction: Meet to the west of you, at Ben & Jerry's ice cream on South 40th Street, on the...
ID 1 | Noun: 'cafe' | Candidates: 47
  ↳ Instruction: Meet me at the cafe north of you on the north side of West Girard Avenue. BB&T b...
ID 2 | Noun: 'historic memorial' | Candidates: 77
  ↳ Instruction: Meet me at the historic memorial on the south side of Arch Street. It is a few s...
ID 3 | Noun: 'cafe' | Candidates: 25
  ↳ Instruction: Go south and a bit east. You'll find me at the cafe across the street from a ban...
ID 4 | Noun: 'American Eagle Outfitters' | Candidates: 45
  ↳ Instruction: I am at the American Eagle Outfitters which is in the middle of the block on Che...

--- ❌ REMAINING CONTRADICTORY (The 'Phantom' Problem) ---
ID 5 | Noun: 'this car sharing place here' | Candidates: 0
  ↳ Reason: Landmark not found in OSM

Mini-Test (5AM, Need Sleep)

In [27]:
import pandas as pd
import os
import config
from src.symbolic_solver import SymbolicSolver
from src.oracle_engine import OracleEngine
import pickle
from tqdm import tqdm

# 1. Setup paths for Manhattan
config.CURRENT_CITY = "manhattan"
graph_path = config.get_graph_path()
poi_path = config.get_poi_path()
json_path = os.path.join(config.BASE_DIR, "data", "manhattan", "manhattan.json")

# 2. Initialize (this takes a few seconds)
print("🚀 Loading Manhattan Engines...")
with open(graph_path, 'rb') as f:
    G = pickle.load(f)
oracle = OracleEngine(G, poi_path)
solver = SymbolicSolver(oracle, search_radius=config.get_success_radius())

# 3. Load first 100 rows
df_raw = pd.read_json(json_path, lines=True).head(100)
mini_results = []

print("🧪 Running Mini-Batch (100 rows)...")
for _, row in tqdm(df_raw.iterrows(), total=100):
    # Map the RVS JSON keys to solver inputs
    instr = row['content']
    # Just grab the first node near the start point for this test
    from scipy.spatial import KDTree
    import numpy as np
    node_ids = list(G.nodes())
    coords = np.array([[G.nodes[n]['y'], G.nodes[n]['x']] for n in node_ids])
    tree = KDTree(coords)
    _, idx = tree.query(row['rvs_start_point'])
    start_node = node_ids[idx]

    res = solver.solve(instr, start_node)
    mini_results.append(res['state'])

# 4. Show the Verdict
series = pd.Series(mini_results)
print("\n📊 Mini-Batch Distribution (Manhattan):")
print(series.value_counts())

# 5. Check if 'Answerable' is the majority
if series.value_counts().get('Answerable', 0) > 50:
    print("\n✅ LOGIC LOOKS SOLID. The Salience Filter is working in NYC.")
else:
    print("\n⚠️  WARNING: High Ambiguity/Contradiction. We might need to tune for Manhattan later.")

🚀 Loading Manhattan Engines...


C:\Users\adan\AppData\Local\Temp\ipykernel_18424\648554717.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  G = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_18424\648554717.py:18: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


📍 Extracting coordinates from manhattan 'centroid' column...
✅ Solver Initialized: Found 1 isolated graph components.
🧪 Running Mini-Batch (100 rows)...


100%|██████████| 100/100 [01:09<00:00,  1.45it/s]


📊 Mini-Batch Distribution (Manhattan):
Ambiguous        56
Answerable       36
Contradictory     8
Name: count, dtype: int64

⚠️  WARNING: High Ambiguity/Contradiction. We might need to tune for Manhattan later.


🧪 Next Attempt: Run the "Calibration Test" With 0.7 Ratio (Instead of 0.5)

In [2]:
import sys
import os

# Get the root directory (one level up from /notebooks)
root_dir = os.path.dirname(os.getcwd())

# Add the root to the system path if it's not already there
if root_dir not in sys.path:
    sys.path.append(root_dir)

import config
print(f"✅ Config loaded successfully from: {root_dir}")

✅ Config loaded successfully from: c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning


In [4]:
import config
import pickle
import os
from src.oracle_engine import OracleEngine
from src.symbolic_solver import SymbolicSolver

# 1. Re-initialize the engines for Manhattan
config.CURRENT_CITY = "manhattan"
graph_path = config.get_graph_path()
poi_path = config.get_poi_path()

print("🚀 Loading Manhattan Engines...")
with open(graph_path, 'rb') as f:
    G = pickle.load(f)

# Initialize the Oracle and the Base Solver
oracle = OracleEngine(G, poi_path)
solver = SymbolicSolver(oracle, search_radius=config.get_success_radius())

print("✅ Solver instance 'solver' is now live.")

🚀 Loading Manhattan Engines...


C:\Users\adan\AppData\Local\Temp\ipykernel_24540\3827108975.py:14: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  G = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_24540\3827108975.py:14: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


📍 Extracting coordinates from manhattan 'centroid' column...
✅ Solver Initialized: Found 1 isolated graph components.
✅ Solver instance 'solver' is now live.


In [5]:
import types
import config

# 1. Define the "Tuned" Salience Filter
def tuned_resolve_ambiguity(self, candidates, start_coords):
    """
    A temporary replacement for the solver's ambiguity logic.
    Uses 0.7 ratio for Manhattan and 0.5 for others.
    """
    if not candidates:
        return None
    
    # Sort candidates by distance from start_node
    # Assuming each cand is {'coords': (lat, lon), 'node_id': ...}
    from scipy.spatial.distance import euclidean
    
    sorted_cands = sorted(
        candidates, 
        key=lambda x: euclidean(start_coords, x['coords'])
    )
    
    if len(sorted_cands) == 1:
        return sorted_cands[0]
    
    d1 = euclidean(start_coords, sorted_cands[0]['coords'])
    d2 = euclidean(start_coords, sorted_cands[1]['coords'])
    
    # --- THE MANHATTAN TUNE ---
    # Philadelphia = 0.5 (Strict) | Manhattan = 0.7 (Dense-aware)
    threshold = 0.7 if config.CURRENT_CITY == "manhattan" else 0.5
    
    if d1 < (d2 * threshold):
        return sorted_cands[0]
    
    return None # Still ambiguous

# 2. Inject (Monkey Patch) it into your existing solver instance
# This replaces the method on the 'solver' object specifically
solver.resolve_ambiguity = types.MethodType(tuned_resolve_ambiguity, solver)

print(f"✅ Solver 'resolve_ambiguity' patched for {config.CURRENT_CITY} (Ratio: 0.7)")

✅ Solver 'resolve_ambiguity' patched for manhattan (Ratio: 0.7)


In [8]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from scipy.spatial import KDTree

# 1. Re-load the Data (First 100 rows)
json_path = os.path.join(root_dir, "data", "manhattan", "manhattan.json")
try:
    df_raw = pd.read_json(json_path, lines=True).head(100)
    print(f"✅ Loaded {len(df_raw)} Manhattan samples.")
except Exception as e:
    print(f"❌ Error loading data: {e}")

# 2. Re-build the Spatial Index
node_ids = list(G.nodes())
coords = np.array([[G.nodes[n]['y'], G.nodes[n]['x']] for n in node_ids])
tree = KDTree(coords)

# 3. Run the Tuned Loop
mini_results_tuned = []

print("🧪 Running Tuned Manhattan Test (0.7 Ratio)...")
for _, row in tqdm(df_raw.iterrows(), total=len(df_raw)):
    instr = row['content']
    
    # Snap GPS to Graph Nodes
    _, idx = tree.query(row['rvs_start_point'])
    start_node = node_ids[idx]

    # This call now uses the patched 0.7 logic from the previous step
    res = solver.solve(instr, start_node)
    mini_results_tuned.append(res['state'])

# 📊 Final Verdict Comparison
tuned_series = pd.Series(mini_results_tuned)
print("\n📊 TUNED Mini-Batch Distribution (Manhattan):")
print(tuned_series.value_counts())

# Reminder of the 5 AM Baseline:
# Ambiguous: 56 | Answerable: 36 | Contradictory: 8

✅ Loaded 100 Manhattan samples.
🧪 Running Tuned Manhattan Test (0.7 Ratio)...


100%|██████████| 100/100 [07:36<00:00,  4.57s/it]


📊 TUNED Mini-Batch Distribution (Manhattan):
Answerable       81
Ambiguous        11
Contradictory     8
Name: count, dtype: int64


Progress with Manhattan. Now let's make sure that this patch doesn't break Philly.

In [10]:
import config
import pandas as pd
import numpy as np
import pickle
import os
import types
from tqdm import tqdm
from scipy.spatial import KDTree, distance

# 1. Switch back to Philly context
config.CURRENT_CITY = "philadelphia"
json_path_philly = os.path.join(root_dir, "data", "philadelphia", "philadelphia.json")

# ADDED lines=True here to fix the ValueError
df_philly = pd.read_json(json_path_philly, lines=True).head(100)

# 2. Re-initialize Philly Engines
print("🚀 Re-loading Philadelphia Engines...")
with open(config.get_graph_path(), 'rb') as f:
    G_philly = pickle.load(f)

# Rebuild Philly Spatial Index
node_ids_philly = list(G_philly.nodes())
coords_philly = np.array([[G_philly.nodes[n]['y'], G_philly.nodes[n]['x']] for n in node_ids_philly])
tree_philly = KDTree(coords_philly)

oracle_philly = OracleEngine(G_philly, config.get_poi_path())
solver_philly = SymbolicSolver(oracle_philly, search_radius=config.get_success_radius())

# 3. Comparison Function
def test_solve(solver, ratio):
    def temp_resolve(self, candidates, start_coords):
        if not candidates: return None
        # Sort by distance
        sorted_cands = sorted(candidates, key=lambda x: distance.euclidean(start_coords, x['coords']))
        if len(sorted_cands) == 1: return sorted_cands[0]
        
        d1 = distance.euclidean(start_coords, sorted_cands[0]['coords'])
        d2 = distance.euclidean(start_coords, sorted_cands[1]['coords'])
        return sorted_cands[0] if d1 < (d2 * ratio) else None
    
    solver.resolve_ambiguity = types.MethodType(temp_resolve, solver)
    
    results = []
    for _, row in df_philly.iterrows():
        _, idx = tree_philly.query(row['rvs_start_point'])
        res = solver.solve(row['content'], node_ids_philly[idx])
        results.append(res['state'])
    return pd.Series(results).value_counts()

# 4. Run Comparison
print("\n⚖️ COMPARING PHILLY RATIOS (100 Samples)")
print("-" * 40)
print("ORIGINAL (0.5 Ratio):")
print(test_solve(solver_philly, 0.5))

print("\nPATCHED (0.7 Ratio):")
print(test_solve(solver_philly, 0.7))

🚀 Re-loading Philadelphia Engines...


C:\Users\adan\AppData\Local\Temp\ipykernel_24540\1899148802.py:20: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G_philly = pickle.load(f)


📍 Extracting coordinates from philadelphia 'centroid' column...
✅ Solver Initialized: Found 1 isolated graph components.

⚖️ COMPARING PHILLY RATIOS (100 Samples)
----------------------------------------
ORIGINAL (0.5 Ratio):
Answerable       79
Contradictory    11
Ambiguous        10
Name: count, dtype: int64

PATCHED (0.7 Ratio):
Answerable       79
Contradictory    11
Ambiguous        10
Name: count, dtype: int64


In [12]:
print("📊 Columns in Parquet:", df_silver.columns.tolist())
df_silver.head(3)

📊 Columns in Parquet: ['sample_id', 'city', 'instruction', 'oracle_label', 'candidate_count', 'start_node', 'gold_goal_node', 'extracted_category', 'extracted_noun', 'target_tags']


,sample_id,city,instruction,oracle_label,candidate_count,start_node,gold_goal_node,extracted_category,extracted_noun,target_tags
0,N/A,philadelphia,"Meet to the west of you, at Ben & Jerry's ice ...",Answerable,69,#6593007625,#2100740222,SHOP,None,None
1,N/A,philadelphia,Meet me at the cafe north of you on the north ...,Answerable,1166,#5723276180,#3275071095,UNKNOWN,None,None
2,N/A,philadelphia,Meet me at the historic memorial on the south ...,Answerable,2581,#1590298601,#4879082921,UNKNOWN,None,None


In [14]:
import pandas as pd
from tqdm import tqdm
import types
from scipy.spatial import distance

# 1. Ensure the DataFrame is named correctly
# (Using the path we discussed earlier)
project_root = root_dir # from our sys.path fix
parquet_path = os.path.join(project_root, "data/philadelphia/philadelphia_silver_standard.parquet")
df_silver = pd.read_parquet(parquet_path)

print(f"✅ Loaded {len(df_silver)} Philly Silver Standard rows.")

# 2. Double-check the 0.7 patch is active on the solver
def tuned_resolve(self, candidates, start_coords):
    if not candidates: return None
    sorted_cands = sorted(candidates, key=lambda x: distance.euclidean(start_coords, x['coords']))
    if len(sorted_cands) == 1: return sorted_cands[0]
    d1 = distance.euclidean(start_coords, sorted_cands[0]['coords'])
    d2 = distance.euclidean(start_coords, sorted_cands[1]['coords'])
    return sorted_cands[0] if d1 < (d2 * 0.7) else None

solver_philly.resolve_ambiguity = types.MethodType(tuned_resolve, solver_philly)

# 3. Run the Regression Test
philly_results_07 = []
for _, row in tqdm(df_silver.iterrows(), total=len(df_silver)):
    res = solver_philly.solve(row['instruction'], row['start_node'])
    philly_results_07.append(res['state'])

# 4. Compare
new_series = pd.Series(philly_results_07)
print("\n📊 NEW Distribution with 0.7 Ratio (Philly):")
print(new_series.value_counts())

✅ Loaded 1278 Philly Silver Standard rows.


  0%|          | 0/1278 [00:00<?, ?it/s]

100%|██████████| 1278/1278 [34:15<00:00,  1.61s/it]


📊 NEW Distribution with 0.7 Ratio (Philly):
Answerable       1035
Contradictory     161
Ambiguous          82
Name: count, dtype: int64


🔍 The Verdict: Stable

### Pittsburgh Threshold Sweep (100 Samples)

In [15]:
import config
import pandas as pd
import numpy as np
import os
import types
from tqdm import tqdm
from scipy.spatial import KDTree

# 1. Switch to Pittsburgh Context
config.CURRENT_CITY = "pittsburgh"
json_path_pitt = os.path.join(root_dir, "data", "pittsburgh", "pittsburgh.json")
df_pitt = pd.read_json(json_path_pitt, lines=True).head(100)

print(f"✅ Loaded 100 Pittsburgh samples.")

# 2. Re-initialize Pittsburgh Engines
with open(config.get_graph_path(), 'rb') as f:
    G_pitt = pickle.load(f)

node_ids_pitt = list(G_pitt.nodes())
coords_pitt = np.array([[G_pitt.nodes[n]['y'], G_pitt.nodes[n]['x']] for n in node_ids_pitt])
tree_pitt = KDTree(coords_pitt)

oracle_pitt = OracleEngine(G_pitt, config.get_poi_path())
solver_pitt = SymbolicSolver(oracle_pitt, search_radius=config.get_success_radius())

# 3. Sweep Function
def run_sweep(target_ratio):
    results = []
    for _, row in df_pitt.iterrows():
        # Snap to graph
        _, idx = tree_pitt.query(row['rvs_start_point'])
        start_node = node_ids_pitt[idx]
        
        # Manually apply the ratio for this test run
        # (This mimics the logic inside your solve method)
        res = solver_pitt.solve(row['content'], start_node)
        
        # We need to manually override the salience check for the sweep 
        # since solver_pitt.solve uses the config value.
        # Let's patch the solver instance for this specific ratio:
        def temp_resolve(self, candidates, start_coords):
            if not candidates: return None
            from scipy.spatial.distance import euclidean
            sorted_cands = sorted(candidates, key=lambda x: euclidean(start_coords, x['coords']))
            if len(sorted_cands) == 1: return sorted_cands[0]
            d1 = euclidean(start_coords, sorted_cands[0]['coords'])
            d2 = euclidean(start_coords, sorted_cands[1]['coords'])
            return sorted_cands[0] if d1 < (d2 * target_ratio) else None
        
        solver_pitt.resolve_ambiguity = types.MethodType(temp_resolve, solver_pitt)
        
        final_res = solver_pitt.solve(row['content'], start_node)
        results.append(final_res['state'])
    return pd.Series(results).value_counts()

# 4. Execute Comparison
ratios = [0.5, 0.6, 0.7]
sweep_results = {}

print("\n⚖️ PITTSBURGH THRESHOLD SWEEP")
print("-" * 40)
for r in ratios:
    print(f"Testing Ratio: {r}...")
    sweep_results[r] = run_sweep(r)

# 📊 Final Results Table
sweep_df = pd.DataFrame(sweep_results).fillna(0).astype(int)
print("\n📊 Summary Table (Pittsburgh):")
print(sweep_df)

✅ Loaded 100 Pittsburgh samples.


C:\Users\adan\AppData\Local\Temp\ipykernel_24540\2335029569.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  G_pitt = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_24540\2335029569.py:18: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G_pitt = pickle.load(f)


📍 Extracting coordinates from pittsburgh 'centroid' column...
✅ Solver Initialized: Found 1 isolated graph components.

⚖️ PITTSBURGH THRESHOLD SWEEP
----------------------------------------
Testing Ratio: 0.5...
Testing Ratio: 0.6...
Testing Ratio: 0.7...

📊 Summary Table (Pittsburgh):
               0.5  0.6  0.7
Answerable      81   81   81
Contradictory   16   16   16
Ambiguous        3    3    3


📉 Analysis of the Table
Zero Sensitivity: The fact that the numbers are identical across all three ratios means your Pittsburgh POIs are well-distributed.

Ambiguity is Low: With only 3 Ambiguous cases, Pittsburgh doesn't suffer from the "City of Coffee Shops" problem that Manhattan does.

The Verdict: We can safely keep Pittsburgh at 0.5. It follows the standard "Suburban/Mid-sized City" logic perfectly.

🔍 Manhattan "Contradictory" Audit

In [18]:
import os
manhattan_dir = os.path.join(root_dir, "data", "manhattan")
print("📂 Files found in Manhattan folder:", os.listdir(manhattan_dir))

📂 Files found in Manhattan folder: ['appendix_rescued_samples_v4.csv', 'manhattan.json', 'manhattan_geo_paths.gpkg', 'manhattan_graph.gpickle', 'manhattan_poi.pkl', 'manhattan_silver_standard.parquet', 'manhattan_silver_standard_V4.parquet', 'manhattan_streets.pkl', 'underspecified_variants_old.json']


In [19]:
import pandas as pd
import os

# 1. Load the verified file
manhattan_path = os.path.join(root_dir, "data", "manhattan", "manhattan_silver_standard.parquet")
df_mh = pd.read_parquet(manhattan_path)

# 2. Filter for Contradictory cases
# (Using .head(10) to give you a good variety)
df_fail = df_mh[df_mh['oracle_label'] == 'Contradictory'].head(10)

print(f"🕵️ Auditing 10 'Contradictory' Samples...\n")
print(f"{'INSTRUCTION':<60} | {'CATS':<5} | {'REASON'}")
print("-" * 100)

for _, row in df_fail.iterrows():
    instr = (row['instruction'][:57] + '..') if len(row['instruction']) > 57 else row['instruction']
    cats = row['candidate_count']
    
    reason = "🚫 NO LANDMARK FOUND" if cats == 0 else "🧭 GEOMETRIC MISMATCH"
    
    print(f"{instr:<60} | {cats:<5} | {reason}")

# 3. Deep Dive into one 'Geometric Mismatch' if available
mismatch = df_mh[(df_mh['oracle_label'] == 'Contradictory') & (df_mh['candidate_count'] > 0)].head(1)
if not mismatch.empty:
    print("\n🔍 Deep Dive into a Geometric Mismatch:")
    print(f"Instruction: {mismatch.iloc[0]['instruction']}")
    print(f"Category: {mismatch.iloc[0]['extracted_category']}")
    print("Interpretation: The landmark exists, but it failed the directional or reachability check.")

🕵️ Auditing 10 'Contradictory' Samples...

INSTRUCTION                                                  | CATS  | REASON
----------------------------------------------------------------------------------------------------
I wanna try out the Philly Pretzel Factory on Chambers St..  | 0     | 🚫 NO LANDMARK FOUND
Meet me for coffee at the cafe. You'll want to go northea..  | 0     | 🚫 NO LANDMARK FOUND
Meet me at the toilets. Go north for a while until you re..  | 0     | 🚫 NO LANDMARK FOUND
I'm  at the parking entrence on 61st street.  You'll have..  | 0     | 🚫 NO LANDMARK FOUND
Meet me at the thai fast food restaurant south of you on ..  | 0     | 🚫 NO LANDMARK FOUND
Let's get away from the crazy people in the park and go r..  | 0     | 🚫 NO LANDMARK FOUND
go straight along the road . and the green is startibg po..  | 0     | 🚫 NO LANDMARK FOUND
I'm at the tapas restaurant next to the bar. To get here,..  | 0     | 🚫 NO LANDMARK FOUND
I'm northeast of you in the Lenox Hill neighborhoo

In [20]:
import pandas as pd
import pickle
import os

# 1. Load the Manhattan POI data directly to see what the Oracle sees
poi_path = os.path.join(root_dir, "data", "manhattan", "manhattan_poi.pkl")
with open(poi_path, 'rb') as f:
    df_poi = pickle.load(f)

print(f"📊 Total POIs in Manhattan database: {len(df_poi)}")

# 2. Search for "Pretzel" anywhere in the name (Case-Insensitive)
search_term = "Pretzel"
results = df_poi[df_poi['name'].str.contains(search_term, case=False, na=False)]

if not results.empty:
    print(f"\n✅ Found {len(results)} matches for '{search_term}':")
    print(results[['name', 'amenity', 'shop']].head(10))
else:
    print(f"\n❌ Zero results for '{search_term}' in the entire Manhattan POI file.")

# 3. Check for "Chambers St" specifically to see the local density
street_matches = df_poi[df_poi['name'].str.contains("Chambers", case=False, na=False)]
print(f"\n📍 Found {len(street_matches)} landmarks mentioning 'Chambers'.")

# 4. Try the Oracle's internal fuzzy logic directly
try:
    # We'll use a dummy start node or just call the name resolver
    # Let's see if the Oracle can find it globally
    print("\n🕵️ Testing Oracle's fuzzy resolver...")
    found_node = oracle_manhattan.resolve_landmark("Philly Pretzel Factory")
    if found_node:
        print(f"🎯 Oracle actually found it! Node ID: {found_node}")
    else:
        print("🤷 Oracle fuzzy search also came up empty.")
except NameError:
    print("⚠️ 'oracle_manhattan' not initialized in this session, skipping step 4.")

C:\Users\adan\AppData\Local\Temp\ipykernel_24540\188873179.py:8: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  df_poi = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_24540\188873179.py:8: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  df_poi = pickle.load(f)


📊 Total POIs in Manhattan database: 20979

✅ Found 2 matches for 'Pretzel':
                        name amenity    shop
6927  Philly Pretzel Factory     NaN     NaN
7130                 PRETZEL     NaN  bakery

📍 Found 15 landmarks mentioning 'Chambers'.

🕵️ Testing Oracle's fuzzy resolver...
⚠️ 'oracle_manhattan' not initialized in this session, skipping step 4.


In [23]:
# 1. Let's see exactly what we're working with
print("📋 Columns in df_poi:", df_poi.columns.tolist())

# 2. Get the Pretzel Factory row (Index 6927)
pretzel_row = df_poi.loc[6927]
print(f"\n🥨 Raw Pretzel Data:\n{pretzel_row}")

# 3. Manually extract based on what we see in the print above
# If it's a 'geometry' column (Point object), we'll do this:
if 'geometry' in df_poi.columns:
    p_lat, p_lon = pretzel_row['geometry'].y, pretzel_row['geometry'].x
elif 'y' in df_poi.columns:
    p_lat, p_lon = pretzel_row['y'], pretzel_row['x']
else:
    # If the columns are something else, replace these strings:
    p_lat, p_lon = pretzel_row['lat'], pretzel_row['lon'] 

# 4. Same for Chambers St
chambers_sample = df_poi[df_poi['name'].str.contains("Chambers", na=False)].iloc[0]
c_lat, c_lon = (chambers_sample['geometry'].y, chambers_sample['geometry'].x) if 'geometry' in df_poi.columns else (chambers_sample['y'], chambers_sample['x'])

# 5. The Moment of Truth: Distance
from src import utils
dist = utils.haversine_vectorized(p_lat, p_lon, c_lat, c_lon)

print(f"\n--- RESULTS ---")
print(f"🥨 Pretzel Factory: ({p_lat:.5f}, {p_lon:.5f})")
print(f"📍 Chambers Sample:  ({c_lat:.5f}, {c_lon:.5f})")
print(f"📏 Distance:         {dist:.2f} meters")

if dist > 1500:
    print("\n🚫 TARGET OUT OF RANGE: This is why it was 'Contradictory'.")
else:
    print("\n⚠️ DATA GAP: It's in range, but likely lacked the tags needed for Phase A search.")

📋 Columns in df_poi: ['unique_id', 'osmid', 'element_type', 'alt_name', 'ele', 'gnis:Class', 'gnis:County', 'gnis:County_num', 'gnis:ST_alpha', 'gnis:ST_num', 'gnis:id', 'import_uuid', 'is_in', 'name', 'name:azb', 'name:fa', 'name:ja', 'name:ko', 'name:ru', 'name:uk', 'name:zh', 'place', 'geometry', 'highway', 'ref', 'source', 'network', 'operator', 'public_transport', 'railway', 'railway:ref', 'train', 'created_by', 'railway:position', 'barrier', 'payment:cash', 'junction', 'old_ref', 'crossing', 'button_operated', 'tactile_paving', 'traffic_signals:sound', 'direction', 'stop', 'maxspeed', 'segregated', 'bus', 'bicycle', 'historic', 'man_made', 'surveillance:type', 'traffic_calming', 'cycleway', 'crossing:island', 'subway', 'wheelchair', 'fixme', 'note', 'name:etymology:wikidata', 'amenity', 'iata', 'brand', 'charge', 'fee', 'manufacturer', 'material', 'surveillance', 'toll', 'website', 'alt_name:pt', 'alt_name:vi', 'importance', 'is_in:continent', 'is_in:country', 'is_in:country_code

Merge the cities:

In [3]:
import pandas as pd
import os
import sys

# 1. Setup paths
root_dir = os.path.dirname(os.getcwd())
if root_dir not in sys.path:
    sys.path.append(root_dir)

data_dir = os.path.join(root_dir, "data")

city_files = {
    "philadelphia": "philadelphia/philadelphia_silver_standard.parquet",
    "pittsburgh": "pittsburgh/pittsburgh_silver_standard.parquet",
    "manhattan": "manhattan/manhattan_silver_standard.parquet"
}

dfs = []

print("📂 Loading and standardizing city files...")
for city, relative_path in city_files.items():
    full_path = os.path.join(data_dir, relative_path.replace("/", os.sep))
    
    if os.path.exists(full_path):
        df = pd.read_parquet(full_path)
        
        # --- TYPE FIX START ---
        # Force sample_id to string to avoid ArrowTypeError
        if 'sample_id' in df.columns:
            df['sample_id'] = df['sample_id'].astype(str)
        # --- TYPE FIX END ---
            
        df['city_source'] = city
        dfs.append(df)
        print(f"✅ Loaded {city}: {len(df)} rows")
    else:
        print(f"❌ Warning: Could not find file for {city} at {full_path}")

# 3. Merge and Save
if dfs:
    # ignore_index=True is good here to create a fresh unified index
    df_master = pd.concat(dfs, ignore_index=True)
    
    output_path = os.path.join(data_dir, "RVS_MASTER_SILVER_STANDARD.parquet")
    
    # We'll use the 'fastparquet' engine or ensure pyarrow is happy
    df_master.to_parquet(output_path, engine='pyarrow')
    
    print("\n" + "="*30)
    print(f"🎉 SUCCESS: Master Dataset Created!")
    print(f"📍 Location: {output_path}")
    print(f"📊 Total Rows: {len(df_master)}")
    
    if 'oracle_label' in df_master.columns:
        print("\n📈 Final Label Breakdown:")
        print(df_master['oracle_label'].value_counts())
else:
    print("🚫 No dataframes were loaded.")

📂 Loading and standardizing city files...
✅ Loaded philadelphia: 1278 rows
✅ Loaded pittsburgh: 1023 rows
✅ Loaded manhattan: 7000 rows

🎉 SUCCESS: Master Dataset Created!
📍 Location: c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\RVS_MASTER_SILVER_STANDARD.parquet
📊 Total Rows: 9301

📈 Final Label Breakdown:
oracle_label
Answerable       7263
Contradictory    1327
Ambiguous         711
Name: count, dtype: int64
